# 💻 Notebook do Aluno — Aula 13: LangGraph — StateGraph, nodes, conditional edges e HITL

**Disciplina:** Prompt Engineering and Artificial Intelligence  
**Instituição:** FIAP — Ciência da Computação · 2026  
**Professor:** Jorge Luiz Gomes  
**Aula 13/14 — Módulo 4 · Grafos de estado em código**  
**1h40min**  
**StateGraph · MemorySaver · HITL · draw_mermaid**  
**Andaime 60%**  

---

## Como usar este notebook

- Rode as células **na ordem**, de cima para baixo (`Shift+Enter`).
- Complete apenas as partes marcadas com `___` e `👉 LACUNA`.
- Não apague o código já pronto — ele é o andaime do lab.
- Salve sua cópia: **Arquivo > Salvar uma cópia no Drive**.

## 📋 Roteiro do Lab

**Lab — Aula 13 · 2º Semestre**  
### Grafo pesquisador com loop condicional para o dominio ★★★

*Grupo 3–4 · 25 minutos · Google Colab*

1. Complete as 5 lacunas — TypedDict com add_messages + tipos; node_buscar com DuckDuckGo; node_responder usando resultados_busca; decidir_continuar com threshold 0.7 e MAX 3; montar o grafo com add_node, add_edge e add_conditional_edges.
2. Visualize o grafo com draw_mermaid() — confirmar que a aresta de volta (avaliar → buscar) aparece no diagrama.
3. Execute 2 perguntas do dominio e observe: quantas iteracoes foram necessarias? O threshold 0.7 foi atingido antes do limite 3 ou precisou do MAX?
4. Experimento com threshold : mudar para 0.5 e 0.9 e documentar a diferenca no numero de iteracoes em celula markdown.

---

## 🧩 Notebook Aluno — 60% de lacunas

Complete as lacunas marcadas com `___`.

In [ ]:
!pip install langgraph langchain-ollama langchain-community duckduckgo-search -q

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Annotated
from pydantic import BaseModel
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_community.tools import DuckDuckGoSearchRun
from IPython.display import Image, display
import os
from google.colab import userdata
os.environ["OLLAMA_HOST"]="https://ollama.com"
os.environ["OLLAMA_API_KEY"]=userdata.get("OLLAMA_API_KEY")

In [ ]:
llm=ChatOllama(model="gpt-oss:120b",temperature=0)

# LACUNA 1: Estado com TypedDict
class Estado(TypedDict):
    mensagens: Annotated[list, ___]  # usar add_messages
    resultados_busca: str
    qualidade: ___  # float
    iteracoes: ___  # int

class Qualidade(BaseModel):
    score:float; suficiente:bool

# LACUNA 2: node_buscar
def node_buscar(state:Estado)->dict:
    bruto=DuckDuckGoSearchRun().run(___)  # query = mensagens[0].content
    return {"resultados_busca":bruto[:1500],"iteracoes":state["iteracoes"]+1}

def node_avaliar(state:Estado)->dict:  # PRONTO
    q=(ChatPromptTemplate.from_template("P:{p}\nR:{r}\nScore:")
       |llm.with_structured_output(Qualidade)).invoke(
        {"p":state["mensagens"][0].content,"r":state["resultados_busca"]})
    return {"qualidade":q.score}

# LACUNA 3: node_responder
def node_responder(state:Estado)->dict:
    resp=llm.invoke([HumanMessage(
        f"P:{state['mensagens'][0].content}\nFonte:{state[___]}"
    )])
    return {"mensagens":[resp]}

# LACUNA 4: decidir_continuar (threshold 0.7, MAX 3)
def decidir_continuar(state:Estado)->str:
    if state["qualidade"]>=___ or state["iteracoes"]>=___:
        return ___  # "responder"
    return ___      # "buscar"

# LACUNA 5: montar grafo
builder=StateGraph(Estado)
builder.add_node("buscar",___);builder.add_node("avaliar",___);builder.add_node("responder",___)
builder.add_edge(START,"buscar");builder.add_edge("buscar",___)  # → avaliar
builder.add_conditional_edges("avaliar",___,{"buscar":"buscar","responder":"responder"})
builder.add_edge("responder",END)
pesquisador=builder.compile(checkpointer=MemorySaver())
try: display(Image(pesquisador.get_graph().draw_mermaid_png()))
except: print(pesquisador.get_graph().draw_mermaid())

---

## ✍️ Suas anotações

Registre aqui as observações pedidas no roteiro (qualidade dos resultados, comparações e conclusões do grupo).

## 📚 Referências da aula

- Docs LangGraph — Guia completo: StateGraph, checkpointing, HITL. langchain-ai.github.io/langgraph/tutorials/introduction
- Docs LangGraph HITL — interrupt_before, update_state, invoke(None). langchain-ai.github.io/langgraph/concepts/human_in_the_loop
- Blog Anthropic Engineering — "Building Effective Agents" (2025). Secao sobre checkpointing e revisao humana. anthropic.com/engineering/building-effective-agents
- Tool Mermaid Live Editor — Para visualizar o output de draw_mermaid() sem instalar playwright. mermaid.live
- Livro Russell, S.; Norvig, P. — Inteligencia Artificial. 3ª ed. Pearson, 2016. Cap. 3 — Resolucao de problemas como busca: base conceitual dos grafos de estado no LangGraph.
- Livro Bornet, P.; Wirtz, J. et al. — Agentic Artificial Intelligence. World Scientific, 2025. A analogia do "funcionário recém-contratado" e HITL como fase de confiança, não trava permanente — a fundamentação por trás do interrupt_before desta aula.
- Livro Gullí, A. — Agentic Design Patterns. O'Reilly, 2025. Cap. 4 — Reflection: o modelo Producer-Critic que justifica separar os nós buscar e avaliar no grafo pesquisador desta aula.

---

**Proxima Aula — Aula 14 (ultima)** — Spec-Driven Development e Encerramento
  
Retrospectiva do semestre · Spec-Driven Development · LangSmith · Proximos passos na carreira · Encerramento.

---

*Copyright © 2026 Prof. Jorge Luiz Gomes · FIAP · Todos os direitos reservados.*